In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/datasets/hasibullahaman/traffic-prediction-dataset/Traffic.csv
/kaggle/input/datasets/hasibullahaman/traffic-prediction-dataset/TrafficTwoMonth.csv


In [9]:
df= pd.read_csv("/kaggle/input/datasets/punitchaurasia98/traffic-dataset/Traffic.csv")

In [10]:
df.head()

,Time,Date,Day of the week,CarCount,BikeCount,BusCount,TruckCount,Total,Traffic Situation
0,12:00:00 AM,10,Tuesday,31,0,4,4,39,low
1,12:15:00 AM,10,Tuesday,49,0,3,3,55,low
2,12:30:00 AM,10,Tuesday,46,0,3,6,55,low
3,12:45:00 AM,10,Tuesday,51,0,2,5,58,low
4,1:00:00 AM,10,Tuesday,57,6,15,16,94,normal


In [16]:
# Normal traffic is a 1.0 multiplier (base speed). Heavy traffic makes a trip take 3x as long.
multiplier_map 
    'normal': 1.5, = {
    'low': 1.0, 
    'high': 2.0, 
    'heavy': 3.0
}
df['Traffic_Multiplier'] = df['Traffic Situation'].map(multiplier_map)


# 2. Translate 'Time' to 'Hour of the Day' (0-23)

df['Hour'] = pd.to_datetime(df['Time'], format='%I:%M:%S %p').dt.hour

# 3. Translate 'Day of the week' to numbers (0-6)
day_map = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 
    'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
}
df['Day_Num'] = df['Day of the week'].map(day_map)

df.sample(5)

,Time,Date,Day of the week,CarCount,BikeCount,BusCount,TruckCount,Total,Traffic Situation,Traffic_Multiplier,Hour,Day_Num
146,12:30:00 PM,11,Wednesday,51,8,11,22,92,normal,1.5,12,2
2823,9:45:00 AM,8,Wednesday,32,10,33,8,83,normal,1.5,9,2
809,10:15:00 AM,18,Wednesday,54,10,30,28,122,high,2.0,10,2
898,8:30:00 AM,19,Thursday,140,34,31,1,206,heavy,3.0,8,3
2745,2:15:00 PM,7,Tuesday,75,24,27,7,133,high,2.0,14,1


In [33]:


X = df[['Hour', 'Day_Num']]
y = df['Traffic_Multiplier']


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.4f}")

Mean Absolute Error (MAE): 0.2546


In [34]:
import pandas as pd
import random

# Generate a synthetic city map (30 roads connecting 20 intersections)
edges = []
for i in range(30):
    node_a = random.randint(1, 20)
    node_b = random.randint(1, 20)
    while node_a == node_b: 
        node_b = random.randint(1, 20)
        
    base_time = random.randint(3, 15) 
    edges.append({'node_a': node_a, 'node_b': node_b, 'base_time': base_time})

graph_df = pd.DataFrame(edges)

# Set the "Simulated Time" (Friday at 5:00 PM)
simulated_hour = 17
simulated_day = 4

# Prepare the exact same columns the model was just trained on
future_conditions = pd.DataFrame({
    'Hour': [simulated_hour] * len(graph_df),
    'Day_Num': [simulated_day] * len(graph_df)
})

# Predict the traffic 
graph_df['traffic_multiplier'] = model.predict(future_conditions)
graph_df['live_travel_time'] = graph_df['base_time'] * graph_df['traffic_multiplier']

# Export for C++
graph_df.to_csv('graph_weights.csv', index=False)
print("Success! graph_weights.csv has been created.")
print(graph_df.head())

Success! graph_weights.csv has been created.
   node_a  node_b  base_time  traffic_multiplier  live_travel_time
0       4       9          8            1.284856         10.278847
1      11       4         14            1.284856         17.987982
2      10      17         10            1.284856         12.848558
3      12       3          7            1.284856          8.993991
4      10       9         15            1.284856         19.272838
